# Implementando makemore

* Qué es un modelo de bigramas estadístico auto-regresivo y cómo implementamos uno?
* Los bigramas son secuencias de dos carácteres, por ejemplo `ab` o `gh`.

**Conceptos clave**
* **Modelo de lenguaje autoregresivo:** genera una secuencia de a un elemento por vez, y cada elemento nuevo se elige según los anteriores. Acá los elementos son caracteres.
* **Modelo de bigramas:** solo mira **el último carácter** para predecir el siguiente, es decir modela $P(c_{t+1} \mid c_t)$.
* **Estadístico / probabilístico:** las probabilidades salen de **contar** cuántas veces aparece cada bigrama en el dataset.
* Usamos el carácter especial `*` para marcar el **inicio** y el **fin** de cada palabra. Así el modelo también aprende con qué letras suele empezar y terminar un nombre.

## Explorando el dataset

In [ ]:
import numpy as np

dataset = [d for d in open('data/city_names_puny.txt', 'r').read().split('\n') if d]  # una ciudad por línea, sin líneas vacías
puny = [x for x in dataset if 'xn--' in x]       # ciudades con caracteres no ASCII (codificadas en punycode)
nopuny = [x for x in dataset if 'xn--' not in x]
len(dataset), len(set(dataset)), len(puny), len(nopuny)   # total, ciudades no repetidas, con y sin punycode

In [ ]:
# Conjunto de caracteres distintos que aparecen en el dataset (dos formas equivalentes)
s = set()
for nombre in dataset:
    for c in nombre:
        s.add(c)

s2 = set([c for nombre in dataset for c in nombre])   # lo mismo con una list comprehension (más rápido)
len(s), s == s2

### Punycode
Los nombres de ciudades tienen tildes, ñ y caracteres de otros alfabetos. Para trabajar con un vocabulario chico (letras ASCII, dígitos, `-`, `.` y espacio) se codifican en **punycode**, el sistema que usan los nombres de dominio de internet: una palabra con caracteres no ASCII se convierte en `xn--` + una versión ASCII.

In [ ]:
import idna

# def utf8_to_punycode(text: str) -> str:
#     """Encodes a UTF-8 string to its Punycode representation."""
#     return idna.encode(text).decode('ascii')

def punyencode(text: str) -> str:
    """Encodes a UTF-8 string to its Punycode representation, handling spaces by encoding each word separately."""
    
    return " ".join([idna.encode(word).decode('ascii') for word in text.split()])
    
def punydecode(punycode: str) -> str:
    """Decodes a Punycode string back to UTF-8."""
    #return idna.decode(punycode)
    return " ".join([idna.decode(word) for word in punycode.split()])

def process_name(name):
    name = name.lower()
    for n in name.split():
        if len(n) < 2:
            return ''
    try:
        return punyencode(name)
    except:
        #print(f'Cant convert {name}')
        return ''

In [ ]:
print(punyencode('pértile'))
print(punydecode('xn--prtile-bva'))   # no cualquier cosa que empieza con xn-- (marca unicode) se puede decodificar

## Generando bigramas

In [ ]:
for d in ['la plata'] + dataset[:3]:   # todos los bigramas de 'la plata' y de las primeras 3 ciudades
    example = ['*'] + list(d) + ['*']
    # zip(example, example[1:]) junta cada caracter con el siguiente: ('*','l'), ('l','a'), ..., ('a','*')
    for c1, c2 in zip(example, example[1:]):
        print(c1, c2)

## Contando bigramas

In [ ]:
charset = ['*'] + sorted(list(set([y for x in dataset for y in x])))
ctoi = {c: i for i, c in enumerate(charset)}   # caracter -> índice
itoc = {i: c for i, c in enumerate(charset)}   # índice -> caracter
len(charset)

In [ ]:
ctoi['3']   # el '3' está en la posición 7

In [ ]:
itoc

In [ ]:
# Matriz de 40x40 donde bigram_count[i, j] = cantidad de veces que el caracter j sigue al caracter i.
# np.zeros crea floats, por eso la convertimos a int.
bigram_count = np.zeros((len(charset), len(charset))).astype(int)

for d in dataset:   # contamos los bigramas de TODO el dataset
    example = ['*'] + list(d) + ['*']
    for c1, c2 in zip(example, example[1:]):
        bigram_count[ctoi[c1], ctoi[c2]] += 1
bigram_count

## Usando una matriz para almacenar las ocurrencias

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(16, 16))
plt.imshow(bigram_count, cmap='Blues')   # más oscuro = bigrama más frecuente
for i in range(len(charset)):
    for j in range(len(charset)):
        chars = itoc[i] + itoc[j]
        plt.text(j, i, chars, ha='center', va='bottom', color='grey', fontsize=8)
        plt.text(j, i, bigram_count[i, j], ha='center', va='top', color='grey', fontsize=8)
plt.axis('off');

## Convertir la matriz de cuentas en una matriz de probabilidades

Cada **fila** `i` de la matriz tiene las cuentas de qué caracteres siguen a `i`. Si dividimos la fila por su suma obtenemos una **distribución de probabilidad**: $P(c_{t+1}=j \mid c_t=i) = \frac{N_{ij}}{\sum_k N_{ik}}$. La fila 0 (`*`) nos dice con qué letra suele **empezar** una ciudad.

In [ ]:
p = bigram_count[0] / bigram_count[0].sum()   # probabilidad de cada caracter de ser el PRIMERO de una ciudad
p

In [ ]:
p[ctoi['a']]   # probabilidad de que una ciudad empiece con 'a'

In [ ]:
np.random.choice(charset, 1, p=p, replace=True)   # sampleamos un caracter inicial según esa distribución

In [ ]:
P = bigram_count / bigram_count.sum(axis=1, keepdims=True)   # matriz de probabilidades: cada fila suma 1
P.sum(axis=1)[:5]

## Sampleando del modelo
Para **generar** una ciudad nueva:
1. Empezamos en `*`.
2. Sampleamos el siguiente caracter usando la fila de `P` del caracter actual.
3. Repetimos hasta que salga `*` (fin de palabra).

Es "autoregresivo" porque cada caracter generado es la entrada del paso siguiente.

In [ ]:
def generate(P):
    out = []
    ix = 0   # empezamos en '*'
    while True:
        ix = np.random.choice(len(charset), p=P[ix])   # siguiente caracter dado el actual
        if ix == 0:                                    # salió '*': terminó la palabra
            break
        out.append(itoc[ix])
    return ''.join(out)

np.random.seed(42)
for i in range(10):
    print(generate(P))

Para comparar, un modelo **sin entrenar** que elige cualquier caracter con la misma probabilidad (1/40) genera basura:

In [ ]:
nclasses = len(charset)
P_uniforme = np.ones((nclasses, nclasses)) / nclasses
for i in range(5):
    print(generate(P_uniforme))

## La Negative Log Likelihood loss (NLL loss)

¿Cómo medimos con **un solo número** qué tan bueno es el modelo?

* **Likelihood (verosimilitud):** la probabilidad que el modelo le asigna a todo el dataset = el **producto** de las probabilidades de cada bigrama. Cuanto más alta, mejor el modelo.
* El producto de muchísimas probabilidades (< 1) da un número diminuto, así que usamos el **logaritmo**: $\log(a \cdot b) = \log a + \log b$. La **log likelihood** es la suma de los logs.
* Como $\log p \le 0$, la log likelihood es negativa y queremos **maximizarla**. Por convención, en ML **minimizamos** una pérdida (loss), así que le cambiamos el signo: **negative log likelihood**.
* Usamos el **promedio** en lugar de la suma para que no dependa del tamaño del dataset:

$$NLL = -\frac{1}{N}\sum_{i=1}^{N} \log P(c_{i+1} \mid c_i)$$

Referencias:
* Modelo perfecto (siempre asigna probabilidad 1 al caracter correcto): $NLL = 0$.
* Modelo uniforme (1/40 a cada caracter): $NLL = -\log(1/40) = \log 40 \approx 3{,}69$.

In [ ]:
log_likelihood = 0.0
n = 0
for d in dataset:
    example = ['*'] + list(d) + ['*']   # incluimos los '*' de inicio y fin
    for c1, c2 in zip(example, example[1:]):
        prob = P[ctoi[c1], ctoi[c2]]
        log_likelihood += np.log(prob)
        n += 1

nll = -log_likelihood / n
print(f'log likelihood: {log_likelihood:.1f}')
print(f'NLL promedio:   {nll:.4f}')
print(f'NLL uniforme:   {np.log(nclasses):.4f}   # modelo que elige al azar')

## Model Smoothing
Problema: si un bigrama **nunca** apareció en el dataset, su probabilidad es 0 y $\log 0 = -\infty$. Cualquier palabra nueva que lo contenga tendría una NLL **infinita**.

**Solución (smoothing):** sumar una cuenta "falsa" a todos los bigramas antes de normalizar, por ejemplo `+1` (Laplace smoothing). Así ninguna probabilidad es 0.
* Con un valor chico (1) el modelo casi no cambia.
* Con un valor enorme (por ejemplo `+1000000`) todas las cuentas quedan parecidas y el modelo se vuelve **uniforme** (genera cosas al azar).

In [ ]:
def nll_de(P, palabras):
    ll, n = 0.0, 0
    for d in palabras:
        example = ['*'] + list(d) + ['*']
        for c1, c2 in zip(example, example[1:]):
            ll += np.log(P[ctoi[c1], ctoi[c2]])
            n += 1
    return -ll / n

# Armamos una "palabra rara" con un bigrama que NUNCA aparece en el dataset (cuenta 0, sin usar '*')
i, j = [(i, j) for i, j in np.argwhere(bigram_count == 0) if i != 0 and j != 0][0]
rara = 'ca' + itoc[i] + itoc[j] + 'a'
print(f'palabra rara: {rara!r}  (el bigrama {itoc[i] + itoc[j]!r} aparece {bigram_count[i, j]} veces)')

with np.errstate(divide='ignore'):
    print('sin smoothing:', nll_de(P, [rara]))   # inf: tiene un bigrama con probabilidad 0

for k in [1, 100, 1_000_000]:
    P_s = (bigram_count + k) / (bigram_count + k).sum(axis=1, keepdims=True)
    print(f'smoothing +{k}: NLL dataset = {nll_de(P_s, dataset):.4f}, palabra rara = {nll_de(P_s, [rara]):.4f}')

## Conclusiones
* En este módulo creamos un modelo de bigramas probabilistico autoregresivo.
* El modelo cuenta las ocurrencias de los bigramas y luego calcula las distribuciones de probabilidad de un carácter dado el carácter anterior.
* Toda la información está contenida en una matriz de probabilidades
* Vimos como calcular que tan bueno o malo es el modelo respecto a nuestro universo de carácteres conocidos (charset) y a un modelo que elige carácteres al azar.
* Esta métrica nos va a acompañar el resto del curso y vamos a volver sobre esto luego.

## Ejercicios
* Generar un modelo de trigramas (intermedio).
  * Pista: En lugar de una matriz, guardar las probabilidades en un tensor tridimensional.